#Pattern Recognition Mini Project - Animal Sound Classification - Kelompok 3


##1. Load Library dan Dataset

In [1]:
!pip install datasets soundfile librosa -q

In [2]:
import numpy as np
import librosa
import soundfile as sf
from io import BytesIO
from datasets import load_dataset, concatenate_datasets, Dataset, Audio

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

In [3]:
def load_n_samples(name, label, n=400):
    stream = load_dataset("cgeorgiaw/animal-sounds", name, streaming=True)["train"]
    samples = []
    for i, example in enumerate(stream):
        if i >= n:
            break
        samples.append({
            "audio": example["audio"],
            "label": label
        })
    return Dataset.from_list(samples)

birds       = load_n_samples("birds",       "birds")
macaque     = load_n_samples("macaque",     "macaque")
giant_otter = load_n_samples("giant_otter", "giant_otter")
orca        = load_n_samples("orca",        "orca")
zebra_finch = load_n_samples("zebra_finch", "zebra_finch")

# Gabungkan
dataset = concatenate_datasets([birds, macaque, giant_otter, orca, zebra_finch])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/688 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/688 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/688 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/883 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/688 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/595 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/688 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/3406 [00:00<?, ?it/s]

In [4]:
print(f"Total sampel: {len(dataset)}")
print(f"Kolom      : {dataset.column_names}")

Total sampel: 2000
Kolom      : ['audio', 'label']


##2. Preprocessing


###2A. Resampling

In [5]:
# Cast audio supaya bisa di-decode
dataset = dataset.cast_column("audio", Audio())

# Cek sample rate tiap kelas
for i, label in zip([0, 400, 800, 1200, 1600], ["birds", "macaque", "giant_otter", "orca", "zebra_finch"]):
    sr = dataset[i]["audio"]["sampling_rate"]
    print(f"{label:20s} → sample rate: {sr} Hz")

birds                → sample rate: 44100 Hz
macaque              → sample rate: 24414 Hz
giant_otter          → sample rate: 96000 Hz
orca                 → sample rate: 44100 Hz
zebra_finch          → sample rate: 44100 Hz


In [6]:
# Resample semua ke 22050 Hz
dataset = dataset.cast_column("audio", Audio(sampling_rate=22050))

# Verifikasi
for i, label in zip([0, 400, 800, 1200, 1600], ["birds", "macaque", "giant_otter", "orca", "zebra_finch"]):
    sr = dataset[i]["audio"]["sampling_rate"]
    print(f"{label:20s} → sample rate: {sr} Hz")

birds                → sample rate: 22050 Hz
macaque              → sample rate: 22050 Hz
giant_otter          → sample rate: 22050 Hz
orca                 → sample rate: 22050 Hz
zebra_finch          → sample rate: 22050 Hz


###2B. Stereo ke Mono

In [7]:
for i, label in zip([0, 400, 800, 1200, 1600], ["birds", "macaque", "giant_otter", "orca", "zebra_finch"]):
    audio = np.array(dataset[i]["audio"]["array"])
    print(f"{label:20s} → shape: {audio.shape}")

birds                → shape: (110336,)
macaque              → shape: (13573,)
giant_otter          → shape: (7714,)
orca                 → shape: (88200,)
zebra_finch          → shape: (4673,)


-> Semua audio sudah mono, tidak perlu konversi stereo ke mono.

###2C. Trim Silence

In [8]:
def trim_silence(example):
    audio = np.array(example["audio"]["array"], dtype=np.float32)
    audio_trimmed, _ = librosa.effects.trim(audio, top_db=20)

    # Simpan sebagai bytes supaya tidak error saat encode
    buffer = BytesIO()
    sf.write(buffer, audio_trimmed, 22050, format="wav")
    example["audio"] = {"bytes": buffer.getvalue(), "path": None}
    return example

dataset = dataset.map(trim_silence)

# Verifikasi
from datasets import Audio
dataset = dataset.cast_column("audio", Audio(sampling_rate=22050))

for i, label in zip([0, 400, 800, 1200, 1600], ["birds", "macaque", "giant_otter", "orca", "zebra_finch"]):
    audio = np.array(dataset[i]["audio"]["array"])
    duration = len(audio) / 22050
    print(f"{label:20s} → {len(audio)} sampel ({duration:.2f} detik)")

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

birds                → 110336 sampel (5.00 detik)
macaque              → 13573 sampel (0.62 detik)
giant_otter          → 5120 sampel (0.23 detik)
orca                 → 88200 sampel (4.00 detik)
zebra_finch          → 4673 sampel (0.21 detik)


###2D. Filter Durasi

In [9]:
#Cek statistik
durations = []
for example in dataset:
    audio = np.array(example["audio"]["array"])
    duration = len(audio) / 22050
    durations.append(duration)

durations = np.array(durations)
print(f"Total sampel       : {len(durations)}")
print(f"Durasi min         : {durations.min():.2f} detik")
print(f"Durasi max         : {durations.max():.2f} detik")
print(f"Durasi rata-rata   : {durations.mean():.2f} detik")
print(f"Sampel > 3 detik   : {(durations > 3.0).sum()}")
print(f"Sampel <= 3 detik  : {(durations <= 3.0).sum()}")

Total sampel       : 2000
Durasi min         : 0.04 detik
Durasi max         : 9.40 detik
Durasi rata-rata   : 1.96 detik
Sampel > 3 detik   : 706
Sampel <= 3 detik  : 1294


In [10]:
#Cek jumlah sampel yang lebih dari 3 detik
for i, label in zip(
    [(0,400), (400,800), (800,1200), (1200,1600), (1600,2000)],
    ["birds", "macaque", "giant_otter", "orca", "zebra_finch"]
):
    start, end = i
    count = 0
    for j in range(start, end):
        audio = np.array(dataset[j]["audio"]["array"])
        duration = len(audio) / 22050
        if duration > 3.0:
            count += 1
    print(f"{label:20s} → {count}/400 sampel > 3 detik")

birds                → 299/400 sampel > 3 detik
macaque              → 0/400 sampel > 3 detik
giant_otter          → 4/400 sampel > 3 detik
orca                 → 399/400 sampel > 3 detik
zebra_finch          → 4/400 sampel > 3 detik


In [11]:
#Cek jumlah sampel dengan threshold tertentu
thresholds = [0.5, 1.0, 1.5, 2.0]

for thresh in thresholds:
    print(f"\n--- Threshold > {thresh} detik ---")
    for i, label in zip(
        [(0,400), (400,800), (800,1200), (1200,1600), (1600,2000)],
        ["birds", "macaque", "giant_otter", "orca", "zebra_finch"]
    ):
        start, end = i
        count = sum(
            1 for j in range(start, end)
            if len(np.array(dataset[j]["audio"]["array"])) / 22050 > thresh
        )
        print(f"  {label:20s} → {count}/400 sampel")


--- Threshold > 0.5 detik ---
  birds                → 399/400 sampel
  macaque              → 320/400 sampel
  giant_otter          → 187/400 sampel
  orca                 → 400/400 sampel
  zebra_finch          → 57/400 sampel

--- Threshold > 1.0 detik ---
  birds                → 398/400 sampel
  macaque              → 0/400 sampel
  giant_otter          → 69/400 sampel
  orca                 → 400/400 sampel
  zebra_finch          → 44/400 sampel

--- Threshold > 1.5 detik ---
  birds                → 390/400 sampel
  macaque              → 0/400 sampel
  giant_otter          → 29/400 sampel
  orca                 → 399/400 sampel
  zebra_finch          → 28/400 sampel

--- Threshold > 2.0 detik ---
  birds                → 380/400 sampel
  macaque              → 0/400 sampel
  giant_otter          → 11/400 sampel
  orca                 → 399/400 sampel
  zebra_finch          → 21/400 sampel


-> Filter durasi tidak dilakukan karena dengan Threshold 1.0 detik, ada kelas yang tidak memiliki sampel. Sementara jika diset ke 0.5 detik, filter kurang informatif.

###(+) Dataset Splitting (Train and Test)

In [12]:
X = [d["audio"] for d in dataset]
y = [d["label"] for d in dataset]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

###2E. Fixing Duration (Padding/Truncation)

-> Pad/trim dilakukan ke 2 detik karena rata-rata durasi adalah 1.96 detik sehingga sampel yang lebih pendek tidak akan terlalu banyak padding-nya dan yang lebih panjang hanya dipotong sedikit.

In [13]:
TARGET_SR      = 22050
TARGET_SAMPLES = TARGET_SR * 2  # 2 detik = 44100 sampel

import soundfile as sf
from io import BytesIO

def pad_trim(audio_dict: dict) -> dict:
    audio = np.array(audio_dict["array"], dtype=np.float32)

    if len(audio) < TARGET_SAMPLES:
        # Pad dengan nol di akhir
        audio = np.pad(audio, (0, TARGET_SAMPLES - len(audio)))
    else:
        # Trim dari depan
        audio = audio[:TARGET_SAMPLES]

    buffer = BytesIO()
    sf.write(buffer, audio, TARGET_SR, format="wav")
    return {"bytes": buffer.getvalue(), "path": None}

X_train = [pad_trim(x) for x in X_train]
X_test  = [pad_trim(x) for x in X_test]

# Verifikasi
print("Verifikasi pad_trim (Train)")
for x, label in zip(X_train[:3], y_train[:3]):
    audio_bytes = x["bytes"]
    audio, sr   = sf.read(BytesIO(audio_bytes))
    print(f"label={label} → {len(audio)} sampel ({len(audio)/sr:.2f} detik)")



Verifikasi pad_trim (Train)
label=giant_otter → 44100 sampel (2.00 detik)
label=birds → 44100 sampel (2.00 detik)
label=giant_otter → 44100 sampel (2.00 detik)


###2F. Normalisasi

In [14]:
def normalize(audio_dict: dict) -> dict:
    audio, sr = sf.read(BytesIO(audio_dict["bytes"]))
    audio     = audio.astype(np.float32)

    max_val = np.max(np.abs(audio))
    if max_val > 0:
        audio = audio / max_val

    buffer = BytesIO()
    sf.write(buffer, audio, sr, format="wav")
    return {"bytes": buffer.getvalue(), "path": None}

X_train = [normalize(x) for x in X_train]
X_test  = [normalize(x) for x in X_test]

# Verifikasi
print("=== Verifikasi normalize (Train) ===")
for x, label in zip(X_train[:3], y_train[:3]):
    audio, _ = sf.read(BytesIO(x["bytes"]))
    print(f"label={label} → min: {audio.min():.4f}, max: {audio.max():.4f}")

=== Verifikasi normalize (Train) ===
label=giant_otter → min: -0.8468, max: 1.0000
label=birds → min: -0.9674, max: 1.0000
label=giant_otter → min: -0.9667, max: 1.0000


## 3. Feature Extraction

### 3A. Mel-Frequency Cepstral Coefficients (MFCC)

In [15]:
def extract_mfcc(audio_dict: dict) -> np.ndarray:
    audio, sr = sf.read(BytesIO(audio_dict["bytes"]))
    audio = audio.astype(np.float32)

    mfcc        = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    mfcc_delta  = librosa.feature.delta(mfcc)
    mfcc_delta2 = librosa.feature.delta(mfcc, order=2)

    mfcc_features = np.concatenate([
        np.mean(mfcc, axis=1),        np.std(mfcc, axis=1),
        np.mean(mfcc_delta, axis=1),  np.std(mfcc_delta, axis=1),
        np.mean(mfcc_delta2, axis=1), np.std(mfcc_delta2, axis=1),
    ])
    return mfcc_features

### 3B. Mel Spectrogram Statistics

In [16]:
def extract_mel(audio_dict: dict) -> np.ndarray:
    audio, sr = sf.read(BytesIO(audio_dict["bytes"]))
    audio = audio.astype(np.float32)

    mel    = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    mel_features = np.concatenate([
        np.mean(mel_db, axis=1),
        np.std(mel_db, axis=1),
    ])
    return mel_features

### 3C. Temporal Features (ZCR, RMS, Spectral Centroid, Bandwith, Rolloff, and Flatness)

In [17]:
def extract_temporal(audio_dict: dict) -> np.ndarray:
    audio, sr = sf.read(BytesIO(audio_dict["bytes"]))
    audio = audio.astype(np.float32)

    zcr       = librosa.feature.zero_crossing_rate(audio)
    rms       = librosa.feature.rms(y=audio)
    centroid  = librosa.feature.spectral_centroid(y=audio, sr=sr)
    bandwidth = librosa.feature.spectral_bandwidth(y=audio, sr=sr)
    rolloff   = librosa.feature.spectral_rolloff(y=audio, sr=sr)
    flatness  = librosa.feature.spectral_flatness(y=audio)

    temporal_features = np.array([
        np.mean(zcr),       np.std(zcr),
        np.mean(rms),       np.std(rms),
        np.mean(centroid),  np.std(centroid),
        np.mean(bandwidth), np.std(bandwidth),
        np.mean(rolloff),   np.std(rolloff),
        np.mean(flatness),  np.std(flatness),
    ])
    return temporal_features

In [18]:
# -- Ekstraksi per split --
mfcc_train     = np.array([extract_mfcc(x)     for x in X_train])
mfcc_test      = np.array([extract_mfcc(x)     for x in X_test])

mel_train      = np.array([extract_mel(x)      for x in X_train])
mel_test       = np.array([extract_mel(x)      for x in X_test])

temporal_train = np.array([extract_temporal(x) for x in X_train])
temporal_test  = np.array([extract_temporal(x) for x in X_test])

# -- Gabungkan semua fitur jadi satu feature matrix --
X_train_features = np.concatenate([mfcc_train, mel_train, temporal_train], axis=1)
X_test_features  = np.concatenate([mfcc_test,  mel_test,  temporal_test],  axis=1)

# ── Verifikasi ───────────────────────────────────────────────────────────────
print("=== Verifikasi Feature Extraction ===")
print(f"X_train_features shape : {X_train_features.shape}")
print(f"X_test_features  shape : {X_test_features.shape}")

print(f"\nMFCC   — train[0] 5 nilai pertama : {mfcc_train[0][:5].round(4)}")
print(f"Mel    — train[0] 5 nilai pertama : {mel_train[0][:5].round(4)}")
print(f"Temporal — train[0] semua nilai   : {temporal_train[0].round(4)}")

=== Verifikasi Feature Extraction ===
X_train_features shape : (1600, 508)
X_test_features  shape : (400, 508)

MFCC   — train[0] 5 nilai pertama : [-6.736628e+02  1.001230e+01 -8.155300e+00 -4.048000e-01 -1.112000e-01]
Mel    — train[0] 5 nilai pertama : [-77.5821 -77.6651 -76.8738 -75.9599 -75.4599]
Temporal — train[0] semua nilai   : [1.630000e-02 5.260000e-02 8.100000e-03 3.090000e-02 2.764497e+02
 9.815654e+02 1.948112e+02 6.158998e+02 4.354905e+02 1.476610e+03
 8.972000e-01 3.026000e-01]


## Machine Learning

### 4A. Feature Vector Construction


In [19]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

In [20]:
print(f"X_train_features : {X_train_features.shape}")
print(f"X_test_features  : {X_test_features.shape}")
print(f"y_train          : {len(y_train)}")
print(f"y_test           : {len(y_test)}")

X_train_features : (1600, 508)
X_test_features  : (400, 508)
y_train          : 1600
y_test           : 400


### 4B. Splitting and Feature Scaling

In [21]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_features)
X_test_scaled  = scaler.transform(X_test_features)

y_train_arr = np.array(y_train)
y_test_arr  = np.array(y_test)

### 4C. Selection of Models

In [22]:
models = {
    "SVM": SVC(),
    "Random Forest": RandomForestClassifier(),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(),
    "MLP": MLPClassifier(max_iter=500)
}

### 4D. Models Evaluation

In [23]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("=" * 65)
print("TAHAP VALIDASI — Stratified 5-Fold Cross Validation (Train Set)")
print("=" * 65)

cv_results = {}

for name, m in models.items():
    scores = cross_validate(
        m, X_train_scaled, y_train_arr,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,   # untuk deteksi overfitting
        n_jobs=-1,
    )

    train_acc = scores["train_score"].mean()
    val_acc   = scores["test_score"].mean()
    val_std   = scores["test_score"].std()
    gap       = train_acc - val_acc   # besar gap = indikasi overfitting

    cv_results[name] = {
        "train_acc": train_acc,
        "val_acc"  : val_acc,
        "val_std"  : val_std,
        "gap"      : gap,
    }

    print(f"\n{name}")
    print(f"  Train Acc (mean)      : {train_acc:.4f}")
    print(f"  Val   Acc (mean ± std): {val_acc:.4f} ± {val_std:.4f}")
    print(f"  Gap (train - val)     : {gap:.4f}  {'[!] potensi overfitting' if gap > 0.1 else '[✓] aman'}")

print("\n" + "=" * 65)


TAHAP VALIDASI — Stratified 5-Fold Cross Validation (Train Set)

SVM
  Train Acc (mean)      : 0.9970
  Val   Acc (mean ± std): 0.9956 ± 0.0015
  Gap (train - val)     : 0.0014  [✓] aman

Random Forest
  Train Acc (mean)      : 1.0000
  Val   Acc (mean ± std): 0.9925 ± 0.0015
  Gap (train - val)     : 0.0075  [✓] aman

KNN
  Train Acc (mean)      : 0.9884
  Val   Acc (mean ± std): 0.9812 ± 0.0081
  Gap (train - val)     : 0.0072  [✓] aman

Naive Bayes
  Train Acc (mean)      : 0.9117
  Val   Acc (mean ± std): 0.9144 ± 0.0271
  Gap (train - val)     : -0.0027  [✓] aman

Decision Tree
  Train Acc (mean)      : 1.0000
  Val   Acc (mean ± std): 0.9737 ± 0.0098
  Gap (train - val)     : 0.0263  [✓] aman

MLP
  Train Acc (mean)      : 1.0000
  Val   Acc (mean ± std): 0.9931 ± 0.0023
  Gap (train - val)     : 0.0069  [✓] aman



In [24]:
print("EVALUASI FINAL — Test Set")
print("=" * 65)

for name, m in models.items():
    # Fit ulang ke seluruh train set (bukan sisa fold terakhir)
    m.fit(X_train_scaled, y_train_arr)
    pred = m.predict(X_test_scaled)

    test_acc = accuracy_score(y_test_arr, pred)
    val_acc  = cv_results[name]["val_acc"]
    gap_val_test = val_acc - test_acc  # val vs test: seharusnya kecil

    print(f"\n{'─'*40}")
    print(f"Model : {name}")
    print(f"  Val  Acc (CV mean) : {val_acc:.4f}")
    print(f"  Test Acc           : {test_acc:.4f}")
    print(f"  Gap (val - test)   : {gap_val_test:.4f}  {'[!] val terlalu optimis' if gap_val_test > 0.05 else '[✓] konsisten'}")
    print(classification_report(y_test_arr, pred))


EVALUASI FINAL — Test Set

────────────────────────────────────────
Model : SVM
  Val  Acc (CV mean) : 0.9956
  Test Acc           : 0.9975
  Gap (val - test)   : -0.0019  [✓] konsisten
              precision    recall  f1-score   support

       birds       1.00      0.99      0.99        83
 giant_otter       0.99      1.00      0.99        68
     macaque       1.00      1.00      1.00        82
        orca       1.00      1.00      1.00        77
 zebra_finch       1.00      1.00      1.00        90

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400


────────────────────────────────────────
Model : Random Forest
  Val  Acc (CV mean) : 0.9925
  Test Acc           : 0.9925
  Gap (val - test)   : 0.0000  [✓] konsisten
              precision    recall  f1-score   support

       birds       1.00      0.99      0.99        83
 giant_otter       0.99      0.99      0.99    